In [ ]:
"""
Evaluate the saved RF regime classifier on the held-out test set,
per station.  Uses ONLY artifacts already saved by the DMOFF pipeline.
"""
import os, glob
import numpy as np, pandas as pd, joblib
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, cohen_kappa_score, confusion_matrix)

BASE = r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes"
STATIONS = {
    "Asomadilla":      "Asomadilla Results",
    "Aljarafe":        "Aljarafe Results",
    "Bermejales":      "Bermejales Results",
    "Ronda del Valle": "RonsadelVAalle Results",
    "Torneo":          "Torneo Results",
}
TARGET_COL, LAGS = "O3", [1, 2, 3, 24]

def find(pattern, folder):
    hits = glob.glob(os.path.join(BASE, folder, "**", pattern), recursive=True)
    return hits[0] if hits else None

def clf_features(df):
    mc = [c for c in ["DD","NO2","PM10","TMP","VV","CO"] if c in df.columns]
    cf = [c for c in mc + ["hour_sin","hour_cos","month_sin","month_cos"] if c in df.columns]
    f  = cf + [TARGET_COL] + [f"{TARGET_COL}_lag{l}" for l in LAGS]
    return [c for c in f if c in df.columns]

rows = []
for station, folder in STATIONS.items():
    clf_p    = find("regime_classifier.pkl", folder) or find("regime_classifier_rf.pkl", folder)
    scaler_p = find("regime_clf_scaler.pkl", folder)
    test_p   = find("test_predictions_all_models.csv", folder)
    if not (clf_p and scaler_p and test_p):
        print(f"[{station}] missing artifacts"); continue

    clf, sc = joblib.load(clf_p), joblib.load(scaler_p)
    df = pd.read_csv(test_p)
    X  = sc.transform(df[clf_features(df)].values)
    y_true = df["regime"].values.astype(int)   # GMM label = ground truth
    y_pred = clf.predict(X)

    rows.append({
        "Station": station, "N_test": len(df),
        "K": len(np.unique(y_true)),
        "Accuracy (%)":      round(accuracy_score(y_true, y_pred)*100, 2),
        "Balanced Acc. (%)": round(balanced_accuracy_score(y_true, y_pred)*100, 2),
        "Macro-F1":          round(f1_score(y_true, y_pred, average="macro"), 3),
        "Cohen's κ":         round(cohen_kappa_score(y_true, y_pred), 3),
    })
    cm = confusion_matrix(y_true, y_pred)
    labels = sorted(set(y_true) | set(y_pred))
    pd.DataFrame(cm, index=[f"true_{i}" for i in labels],
                     columns=[f"pred_{i}" for i in labels]) \
      .to_csv(os.path.join(BASE, folder,
              f"regime_clf_confusion_{station.replace(' ','_')}.csv"))

df_out = pd.DataFrame(rows)
print(df_out.to_string(index=False))
df_out.to_csv(os.path.join(BASE, "regime_classifier_eval_summary.csv"), index=False)